In [ ]:
from pathlib import Path
import time
from typing import Optional

In [ ]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "backend" / "scripts").exists() and (candidate / "frontend").exists():
            return candidate
    raise FileNotFoundError("Could not find the GradeScope project root from the current notebook path.")


PROJECT_ROOT = find_project_root(Path.cwd())
BACKEND_DIR = PROJECT_ROOT / "backend"
SCRIPTS_DIR = BACKEND_DIR / "scripts"
DATA_DIR = BACKEND_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CAPTURE_DIR = RAW_DIR / "portal_captures"
TEXT_DUMPS_DIR = RAW_DIR / "text_dumps"
SUMMARIES_DIR = DATA_DIR / "summaries"
PROCESSED_DIR = DATA_DIR / "processed"

PATHS = {
    "project_root": PROJECT_ROOT,
    "scripts": SCRIPTS_DIR,
    "captures": CAPTURE_DIR,
    "text_dumps": TEXT_DUMPS_DIR,
    "summaries": SUMMARIES_DIR,
    "processed": PROCESSED_DIR,
}
PATHS

In [ ]:
def clean_filename(text: object) -> str:
    import re
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "_", str(text))
    return cleaned.strip("_").lower()


def absolute_url(href: Optional[str]) -> Optional[str]:
    if not href:
        return None
    if href.startswith("http"):
        return href
    if href.startswith("/"):
        return BASE_URL + href
    return BASE_URL + "/" + href


def extract_visible_text(page) -> str:
    try:
        return page.locator("body").inner_text(timeout=5000)
    except Exception:
        return ""


def save_capture(page, filename_prefix: str) -> dict:
    html_path = CAPTURE_DIR / f"{filename_prefix}.html"
    text_path = TEXT_DUMPS_DIR / f"{filename_prefix}.txt"

    html_path.write_text(page.content(), encoding="utf-8")
    text_path.write_text(extract_visible_text(page), encoding="utf-8")

    return {"html": html_path, "text": text_path}


def list_recent_captures(limit: int = 10) -> list[Path]:
    files = sorted(CAPTURE_DIR.glob("*.html"), key=lambda path: path.stat().st_mtime, reverse=True)
    return files[:limit]

In [ ]:
def wait_for_zabdesk_login(page, timeout_ms: int = 180000) -> None:
    page.wait_for_function(
        """
        () => {
            const links = Array.from(document.querySelectorAll("a"));
            const hasAttendance = links.some(a => a.innerText.includes("View Attendance"));
            const hasCurrentResults = links.some(a => a.innerText.includes("Current Semester Results"));
            const hasPreviousResults = links.some(a => a.innerText.includes("Previous Semesters Result"));
            return hasAttendance && hasCurrentResults && hasPreviousResults;
        }
        """,
        timeout=timeout_ms,
    )


def get_link_href(page, text_value: str) -> str:
    locator = page.locator("a", has_text=text_value).first
    href = locator.get_attribute("href")
    resolved = absolute_url(href)
    if not resolved:
        raise ValueError(f"Could not find link for {text_value}")
    return resolved

In [ ]:
recent_captures = list_recent_captures(limit=10)

capture_index = [
    {
        "file": path.name,
        "size_kb": round(path.stat().st_size / 1024, 2),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }
    for path in recent_captures
]

capture_index

In [ ]:
text_dumps = sorted(TEXT_DUMPS_DIR.glob("*.txt"), key=lambda path: path.stat().st_mtime, reverse=True)

text_dump_index = [
    {
        "file": path.name,
        "size_kb": round(path.stat().st_size / 1024, 2),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }
    for path in text_dumps[:10]
]

text_dump_index